In [ ]:
from pathlib import Path
import sys
import importlib

_CWD = Path.cwd().resolve()
if (_CWD / "HFSS_Horn").is_dir():
    _REPO_DIR = _CWD
    _HORN_DIR = _CWD / "HFSS_Horn"
else:
    _HORN_DIR = _CWD
    _REPO_DIR = _CWD.parent

for _path in (_REPO_DIR, _HORN_DIR):
    _path_str = str(_path)
    if _path.exists() and _path_str not in sys.path:
        sys.path.insert(0, _path_str)

import os
import json
import time
import numpy as np
import pandas as pd

import lib_config as config
import lib_backbone
lib_backbone = importlib.reload(lib_backbone)
import lib_plot as plot

%load_ext autoreload
%autoreload 2


In [ ]:
config_path = _HORN_DIR / "_config.toml"
if not config_path.exists():
    raise FileNotFoundError(f"HFSS Horn config not found: {config_path}")

_config = config._loadConfig(config_path)
app_config = config.initParams(_config, debug=True)
cfg = app_config.hfss

backbone = lib_backbone.Backbone(config=app_config)
base_dir = app_config.env.dir_base
backbone.initStorer()

model_paths, model_paths_str = backbone._get_path_models()


In [ ]:
ROUND_DECIMALS = app_config.runtime.round_decimals
RESULTS_FILE = str(backbone._get_dir_run() / Path(_config["io"]["filename_output"]))
LEGACY_TEMP_FILE = str(backbone._get_dir_run() / Path(_config["io"].get("filename_temp", "temp_hfss_export.csv")))
OBJECTIVE_COL = app_config.objective.name
PARAM_NAMES = cfg.param_names
N_REPT = cfg.n_repeats
N_SIML = cfg.n_simulation

TEMP_OUTPUTS = [
    {
        "name": output.name,
        "path": backbone._get_dir_run() / Path(output.filename),
    }
    for output in app_config.io.temp_outputs
]
if not TEMP_OUTPUTS:
    TEMP_OUTPUTS = [{"name": "S11", "path": Path(LEGACY_TEMP_FILE)}]

if N_SIML <= 0:
    raise ValueError("n_simulation must be positive.")


In [ ]:
def wait_for_temp_outputs(temp_outputs, timeout_sec=600, poll_interval_sec=1.0):
    start = time.time()
    paths = [Path(item["path"]) for item in temp_outputs]
    while True:
        if all(path.exists() for path in paths):
            return
        if time.time() - start > timeout_sec:
            raise TimeoutError(f"Timed out waiting for temp outputs: {paths}")
        time.sleep(poll_interval_sec)


def read_temp_output_like_current_s11(temp_hfss_path):
    df_temp = pd.read_csv(temp_hfss_path)
    return df_temp.iloc[-1, -1]


def read_all_temp_outputs(temp_outputs):
    return {
        item["name"]: read_temp_output_like_current_s11(item["path"])
        for item in temp_outputs
    }


def compute_objective(outputs, objective_cfg):
    objective = 0.0
    for term in objective_cfg.terms:
        objective += float(term.weight) * float(outputs[term.column])
    return objective


def round_numeric_row(row, decimals=ROUND_DECIMALS):
    rounded = dict(row)
    for key, value in list(rounded.items()):
        if pd.notna(value) and isinstance(value, (int, float, np.integer, np.floating)):
            rounded[key] = float(np.round(value, decimals))
    return rounded


def getResult(input_params, param_names, temp_outputs, result_file_path):
    wait_for_temp_outputs(temp_outputs)
    header_flag = not os.path.exists(result_file_path)

    outputs = read_all_temp_outputs(temp_outputs)
    result_row = dict(zip(param_names, input_params))
    result_row.update(outputs)
    result_row[OBJECTIVE_COL] = compute_objective(outputs, app_config.objective)
    result_row = round_numeric_row(result_row)

    df_result = pd.DataFrame([result_row])
    df_result.to_csv(result_file_path, mode="a", header=header_flag, index=False)

    for item in temp_outputs:
        try:
            os.remove(item["path"])
        except OSError:
            pass
    return True


In [ ]:
SECTION_FRAC_SLICE = slice(2, 7)
SECTION_FRAC_TOL = 1.0e-8


def section_frac_sum_constraint(x):
    f_wg, f_t1, f_mid, f_t2, f_ap = np.asarray(x, dtype=float).flatten()[SECTION_FRAC_SLICE]
    return f_wg + f_t1 + f_mid + f_t2 + f_ap - 1.0


def enforce_horn_constraints(params, decimals=ROUND_DECIMALS):
    return lib_backbone.enforce_section_frac_constraint(
        params,
        lower_bounds=cfg.lower_bounds,
        upper_bounds=cfg.upper_bounds,
        decimals=decimals,
    )


def assert_horn_constraints(params, tol=SECTION_FRAC_TOL):
    residual = section_frac_sum_constraint(params)
    if abs(residual) > tol:
        raise ValueError(f"section_fracs must sum to 1.0; residual={residual}")
    return True


def round_params(params, decimals=ROUND_DECIMALS):
    params = enforce_horn_constraints(params, decimals=decimals)
    return np.round(params, decimals)


def evaluate_hfss(params):
    params = round_params(params)
    assert_horn_constraints(params)
    sim_id = backbone._getSimulationID()
    backbone.call_subroutine(_config_temp, sim_id, PARAM_NAMES, params, value_fmt=f"{{:.{ROUND_DECIMALS}f}}")
    getResult(params, PARAM_NAMES, TEMP_OUTPUTS, RESULTS_FILE)
    return round_numeric_row(pd.read_csv(RESULTS_FILE).iloc[-1].to_dict())


In [ ]:
_config_temp = {
    "n_simulation": N_SIML,
    "n_repeats": N_REPT,
    "WATCH_DIR": str(backbone._get_dir_run()),
    "INPUT_FILE": str(backbone._get_dir_run() / Path(_config["io"]["filename_input"])),
    "MODEL_FILE": model_paths_str,
    "RESULTS_FILE": RESULTS_FILE,
    "TEMP_FILE": str(Path(LEGACY_TEMP_FILE)),
    "TEMP_OUTPUTS": [{"name": item["name"], "path": str(item["path"])} for item in TEMP_OUTPUTS],
    "DONE_FLAG_FILE": str(backbone._get_dir_run() / Path("hfss.done")),
}

done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
done_flag_path.unlink(missing_ok=True)

with open(base_dir / Path("_config_HFSS.json"), "w") as f:
    json.dump(_config_temp, f, indent=4)

print(f"Temporarily updated {base_dir / Path('_config_HFSS.json')} with run-specific WATCH_DIR for HFSS.")


In [ ]:
active_indices, fixed_point, _ = backbone._buildSamplingIndices(
    dims=cfg.n_params,
    param_groups=cfg.param_groups,
    group_order=getattr(cfg, "group_order", None),
)
assert_horn_constraints(fixed_point)

print("Active Indices:", active_indices)
print("Fixed Point:", np.round(fixed_point, ROUND_DECIMALS))

try:
    for r in range(N_REPT):
        backbone.printn(f"Starting LHS benchmark Repeat {r + 1}/{N_REPT}")
        start = time.perf_counter()

        X_lhs = backbone.LHSsampler_extended(
            dims=cfg.n_params,
            nums=N_SIML,
            lower_bounds=cfg.lower_bounds,
            upper_bounds=cfg.upper_bounds,
            active_indices=active_indices,
            fixed_point=fixed_point,
        )
        X_lhs = np.vstack([round_params(x) for x in X_lhs])
        for x in X_lhs:
            assert_horn_constraints(x)

        visible_history = []
        print(f"{'Iter':<5} | {'New y':<10} | {'Sampler'}")
        for i, params in enumerate(X_lhs, start=1):
            row = evaluate_hfss(params)
            row["Metric"] = np.nan
            row["gamma"] = np.nan
            row["routine_idx"] = i
            row = round_numeric_row(row)
            visible_history.append(row)
            print(f"{i:<5} | {float(np.round(row[OBJECTIVE_COL], ROUND_DECIMALS)):<10} | LHS")

        elapsed = time.perf_counter() - start
        print(f"Repeat {r + 1} completed in {elapsed:.3f} seconds.")

        df_final = pd.DataFrame(visible_history)
        X_train = df_final[PARAM_NAMES].values
        y_train = df_final[OBJECTIVE_COL].values
        best_idx_final = np.argmin(y_train)
        print("-" * 75)
        print("Benchmark Finished.")
        print(f"Best Found: y = {float(np.round(y_train[best_idx_final], ROUND_DECIMALS)):.10f}")
        best_x_str = np.array2string(
            np.round(X_train[best_idx_final], ROUND_DECIMALS),
            precision=ROUND_DECIMALS,
            separator=", ",
        )
        print(f"At location: x = {best_x_str}")

        df_output = backbone._genOutputDataFrame(df_final, objective_col=OBJECTIVE_COL)
        df_output[PARAM_NAMES] = df_output[PARAM_NAMES].round(ROUND_DECIMALS)
        for output in app_config.io.temp_outputs:
            if output.name in df_output:
                df_output[output.name] = df_output[output.name].round(ROUND_DECIMALS)
        if OBJECTIVE_COL in df_output:
            df_output[OBJECTIVE_COL] = df_output[OBJECTIVE_COL].round(ROUND_DECIMALS)
        if "Metric" in df_output:
            df_output["Metric"] = df_output["Metric"].round(ROUND_DECIMALS)
        if "gamma" in df_output:
            df_output["gamma"] = df_output["gamma"].round(ROUND_DECIMALS)

        backbone._addNewDatasetToHDF(df_output, "output", f"repeat_{r + 1}")
        plot.plot_learning_curve(df_output, objective_col=OBJECTIVE_COL)
finally:
    done_flag_path = Path(_config_temp["DONE_FLAG_FILE"])
    done_flag_path.touch()

    json_file = base_dir / Path("_config_HFSS.json")
    json_file.unlink(missing_ok=True)
    if backbone.h5f:
        backbone.h5f.close()
